# 03 — Fast GFS point forecasts for RDU

This notebook replaces the slow GRIB-by-GRIB downloader.

Instead of downloading global GFS fields thousands of times and then keeping one RDU grid point, it uses the **dynamical.org point forecast API** for the NOAA GFS archive.

## What it downloads

Historical training data:
- 40 GFS runs total
- 8 weekly 18Z runs per year for 2021–2025
- July 29 through September 16 in each year
- F010–F120 hourly, then F123–F345 every 3 hours
- **7,440 historical forecast rows**

Final assignment forecast:
- 2026-09-16 18Z
- same variables and lead times

Direct variables:
- temperature_2m
- relative_humidity_2m
- pressure_reduced_to_mean_sea_level
- total_cloud_cover_atmosphere
- wind_u_10m
- wind_v_10m
- precipitation_surface

Derived locally:
- temperature °F
- dew point
- wind speed/direction
- pressure hPa
- precipitation rate mm/hr

## Existing slow checkpoint

You already completed one checkpoint with the old Herbie downloader. This notebook leaves it untouched and, if present, automatically compares overlapping temperature forecasts against it before the bulk download.

Expected old checkpoint path:
`../data/raw/_GFS_HISTORICAL_EXPANDED_CHECKPOINT.csv`

## Outputs

- `../data/raw/NOAA_GFS_FORECAST_HISTORICAL_FAST.csv`
- `../data/raw/NOAA_GFS_FORECAST_FUTURE_FAST.csv`

The API documentation is at https://dynamical.org/api/ .

In [1]:
from pathlib import Path
import math
import random
import time
import json

import numpy as np
import pandas as pd
import requests

## 1. Configuration

In [ ]:
RAW_DIR = Path('../../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

API_BASE = 'https://api.dynamical.org'
DATA_PRODUCT_ID = 'noaa-gfs-forecast'

RDU_LAT = 35.88
RDU_LON = -78.79
LOCAL_TZ = 'America/New_York'

HISTORICAL_YEARS = [2021, 2022, 2023, 2024, 2025]
RUNS_PER_YEAR = 8
FINAL_GFS_RUN = pd.Timestamp('2026-09-16 18:00:00', tz='UTC')

DESIRED_FORECAST_HOURS = list(range(10,121)) + list(range(123,346,3))
MAX_LEAD_TIME_HOURS = max(DESIRED_FORECAST_HOURS)

VARIABLES = [
    'temperature_2m',
    'relative_humidity_2m',
    'pressure_reduced_to_mean_sea_level',
    'total_cloud_cover_atmosphere',
    'wind_u_10m',
    'wind_v_10m',
    'precipitation_surface',
]

BATCH_SIZE = 10  # API maximum is currently 10 queries/request

HIST_CHECKPOINT = RAW_DIR / '_GFS_FAST_HISTORICAL_CHECKPOINT.csv'
FUTURE_CHECKPOINT = RAW_DIR / '_GFS_FAST_FUTURE_CHECKPOINT.csv'
HIST_OUTPUT = RAW_DIR / 'NOAA_GFS_FORECAST_HISTORICAL_FAST.csv'
FUTURE_OUTPUT = RAW_DIR / 'NOAA_GFS_FORECAST_FUTURE_FAST.csv'
OLD_HERBIE_CHECKPOINT = RAW_DIR / '_GFS_HISTORICAL_EXPANDED_CHECKPOINT.csv'

print('Forecast points/run:', len(DESIRED_FORECAST_HOURS))
print('Expected historical rows:', 5 * RUNS_PER_YEAR * len(DESIRED_FORECAST_HOURS))

Forecast points/run: 186
Expected historical rows: 7440


In [3]:
def build_historical_runs():
    runs=[]
    for year in HISTORICAL_YEARS:
        anchor = pd.Timestamp(f'{year}-09-16 18:00:00', tz='UTC')
        for weeks_back in range(RUNS_PER_YEAR-1, -1, -1):
            runs.append(anchor - pd.Timedelta(days=7*weeks_back))
    return runs

HISTORICAL_RUNS = build_historical_runs()

print('Historical runs:', len(HISTORICAL_RUNS))
for i, run in enumerate(HISTORICAL_RUNS, 1):
    print(f'{i:02d}: {run}')

Historical runs: 40
01: 2021-07-29 18:00:00+00:00
02: 2021-08-05 18:00:00+00:00
03: 2021-08-12 18:00:00+00:00
04: 2021-08-19 18:00:00+00:00
05: 2021-08-26 18:00:00+00:00
06: 2021-09-02 18:00:00+00:00
07: 2021-09-09 18:00:00+00:00
08: 2021-09-16 18:00:00+00:00
09: 2022-07-29 18:00:00+00:00
10: 2022-08-05 18:00:00+00:00
11: 2022-08-12 18:00:00+00:00
12: 2022-08-19 18:00:00+00:00
13: 2022-08-26 18:00:00+00:00
14: 2022-09-02 18:00:00+00:00
15: 2022-09-09 18:00:00+00:00
16: 2022-09-16 18:00:00+00:00
17: 2023-07-29 18:00:00+00:00
18: 2023-08-05 18:00:00+00:00
19: 2023-08-12 18:00:00+00:00
20: 2023-08-19 18:00:00+00:00
21: 2023-08-26 18:00:00+00:00
22: 2023-09-02 18:00:00+00:00
23: 2023-09-09 18:00:00+00:00
24: 2023-09-16 18:00:00+00:00
25: 2024-07-29 18:00:00+00:00
26: 2024-08-05 18:00:00+00:00
27: 2024-08-12 18:00:00+00:00
28: 2024-08-19 18:00:00+00:00
29: 2024-08-26 18:00:00+00:00
30: 2024-09-02 18:00:00+00:00
31: 2024-09-09 18:00:00+00:00
32: 2024-09-16 18:00:00+00:00
33: 2025-07-29 18:00

## 2. HTTP helpers

In [4]:
session = requests.Session()
session.headers.update({
    'Accept': 'application/json',
    'Content-Type': 'application/json',
    'User-Agent': 'Duke-AIPI520-RDU-weather-project/1.0',
})


def iso_z(x):
    ts = pd.Timestamp(x)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    return ts.strftime('%Y-%m-%dT%H:%M:%SZ')


def request_json(method, url, *, params=None, payload=None, timeout=180, max_attempts=6):
    for attempt in range(1, max_attempts+1):
        r = session.request(method, url, params=params, json=payload, timeout=timeout)
        if r.status_code < 400:
            return r.json()

        if r.status_code in (429, 503):
            retry_after = r.headers.get('Retry-After')
            try:
                wait = float(retry_after) if retry_after is not None else 2**(attempt-1)
            except ValueError:
                wait = 2**(attempt-1)
            wait += random.uniform(0.2, 1.0)
            print(f'HTTP {r.status_code}; waiting {wait:.1f}s before retry...')
            time.sleep(wait)
            continue

        try:
            detail = r.json()
        except Exception:
            detail = r.text[:2000]
        raise requests.HTTPError(f'HTTP {r.status_code}\nURL: {r.url}\nResponse: {detail}', response=r)

    raise RuntimeError(f'Request failed after {max_attempts} attempts: {url}')

## 3. Verify the product and variables

In [5]:
product = request_json('GET', f'{API_BASE}/v1/data-products/{DATA_PRODUCT_ID}', timeout=60)

print('Product:', product.get('id'))
print('Version:', product.get('version'))
print('Description:', product.get('description'))

meta = product.get('variables', {})
if isinstance(meta, dict):
    available = set(meta.keys())
elif isinstance(meta, list):
    available = {x.get('name') or x.get('id') for x in meta if isinstance(x, dict)}
else:
    available = set()

missing = [v for v in VARIABLES if v not in available]
for v in VARIABLES:
    print(f'{v}:', 'OK' if v in available else 'MISSING')

if missing:
    raise RuntimeError(f'STOP: requested variables missing from product metadata: {missing}')

print('PASS: all requested variables are advertised.')

Product: noaa-gfs-forecast
Version: 0.2.7
Description: Weather forecasts from the Global Forecast System (GFS) operated by NOAA NWS NCEP.
temperature_2m: OK
relative_humidity_2m: OK
pressure_reduced_to_mean_sea_level: OK
total_cloud_cover_atmosphere: OK
wind_u_10m: OK
wind_v_10m: OK
precipitation_surface: OK
PASS: all requested variables are advertised.


## 4. Forecast parsing and derived features

In [6]:
def dewpoint_c(temp_c, rh_pct):
    if pd.isna(temp_c) or pd.isna(rh_pct):
        return np.nan
    rh = float(np.clip(rh_pct, 0.1, 100.0))
    a, b = 17.625, 243.04
    gamma = math.log(rh/100.0) + (a*temp_c)/(b+temp_c)
    return (b*gamma)/(a-gamma)


def wind_speed(u, v):
    return np.nan if pd.isna(u) or pd.isna(v) else math.sqrt(u*u + v*v)


def wind_direction_from(u, v):
    if pd.isna(u) or pd.isna(v):
        return np.nan
    return (270.0 - math.degrees(math.atan2(v, u))) % 360.0


def parse_result(result):
    forecasts = result.get('forecasts', [])
    if len(forecasts) != 1:
        raise ValueError(f'Expected one exact-init forecast; received {len(forecasts)}.')

    fc = forecasts[0]
    leads = fc['leadTimeHours']
    valid_times = fc['validTimes']
    data = fc['data']

    if len(valid_times) != len(leads):
        raise ValueError('validTimes and leadTimeHours lengths differ.')

    for v in VARIABLES:
        if v not in data or len(data[v]) != len(leads):
            raise ValueError(f'Variable missing or wrong length: {v}')

    selected = result.get('point', {}).get('selected', {})
    grid = selected.get('grid', {})
    rows=[]

    for i, lead in enumerate(leads):
        lead=int(lead)
        if lead not in DESIRED_FORECAST_HOURS:
            continue

        def f(v):
            x = data[v][i]
            return np.nan if x is None else float(x)

        tc = f('temperature_2m')
        rh = f('relative_humidity_2m')
        pa = f('pressure_reduced_to_mean_sea_level')
        cloud = f('total_cloud_cover_atmosphere')
        u = f('wind_u_10m')
        v = f('wind_v_10m')
        prate = f('precipitation_surface')

        tf = np.nan if pd.isna(tc) else tc*9/5+32
        dc = dewpoint_c(tc, rh)
        df = np.nan if pd.isna(dc) else dc*9/5+32
        ws = wind_speed(u, v)
        vt = pd.Timestamp(valid_times[i])
        vt = vt.tz_localize('UTC') if vt.tzinfo is None else vt.tz_convert('UTC')

        rows.append({
            'gfs_init_utc': iso_z(fc['initTime']),
            'forecast_hour': lead,
            'valid_time_utc': iso_z(vt),
            'valid_time_local': vt.tz_convert(LOCAL_TZ).isoformat(),
            'gfs_temp_c': tc,
            'gfs_temp_f': tf,
            'gfs_relative_humidity_pct': rh,
            'gfs_dewpoint_c_derived': dc,
            'gfs_dewpoint_f_derived': df,
            'gfs_u10_mps': u,
            'gfs_v10_mps': v,
            'gfs_wind_speed_mps': ws,
            'gfs_wind_speed_mph': np.nan if pd.isna(ws) else ws*2.2369362920544,
            'gfs_wind_direction_deg': wind_direction_from(u, v),
            'gfs_mslp_pa': pa,
            'gfs_mslp_hpa': np.nan if pd.isna(pa) else pa/100.0,
            'gfs_total_cloud_cover_pct': cloud,
            'gfs_precip_rate_kg_m2_s': prate,
            'gfs_precip_rate_mm_hr': np.nan if pd.isna(prate) else prate*3600.0,
            'gfs_grid_lat': selected.get('latitude'),
            'gfs_grid_lon': selected.get('longitude'),
            'gfs_grid_lat_index': grid.get('latitudeIndex'),
            'gfs_grid_lon_index': grid.get('longitudeIndex'),
            'data_product_id': result.get('dataProduct', {}).get('id'),
            'data_product_version': result.get('dataProduct', {}).get('version'),
            'snapshot_id': result.get('snapshotId'),
            'query_id': result.get('queryId'),
        })

    out = pd.DataFrame(rows)
    if len(out) != len(DESIRED_FORECAST_HOURS):
        missing = sorted(set(DESIRED_FORECAST_HOURS)-set(out['forecast_hour']))
        raise RuntimeError(f'Expected {len(DESIRED_FORECAST_HOURS)} rows; got {len(out)}. Missing: {missing[:30]}')
    return out

In [7]:
def query_id_for_run(init_time, prefix='run'):
    ts = pd.Timestamp(init_time)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    return f'{prefix}_{ts.strftime("%Y%m%dT%HZ")}'


def make_query(init_time, prefix='run'):
    return {
        'queryId': query_id_for_run(init_time, prefix),
        'dataProductId': DATA_PRODUCT_ID,
        'location': {'latitude': RDU_LAT, 'longitude': RDU_LON},
        'initTime': iso_z(init_time),
        'maxLeadTimeHours': MAX_LEAD_TIME_HOURS,
        'variables': VARIABLES,
    }

FORECAST_ENDPOINT = f'{API_BASE}/v1/forecasts'
print(json.dumps(make_query(HISTORICAL_RUNS[-1], 'example'), indent=2))

{
  "queryId": "example_20250916T18Z",
  "dataProductId": "noaa-gfs-forecast",
  "location": {
    "latitude": 35.88,
    "longitude": -78.79
  },
  "initTime": "2025-09-16T18:00:00Z",
  "maxLeadTimeHours": 345,
  "variables": [
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_reduced_to_mean_sea_level",
    "total_cloud_cover_atmosphere",
    "wind_u_10m",
    "wind_v_10m",
    "precipitation_surface"
  ]
}


## 5. Preflight: one complete GFS run

In [8]:
payload = {'queries': [make_query(HISTORICAL_RUNS[-1], 'preflight')]}
resp = request_json('POST', FORECAST_ENDPOINT, payload=payload, timeout=180)

if len(resp.get('results', [])) != 1:
    raise RuntimeError('Expected exactly one preflight result.')

preflight_df = parse_result(resp['results'][0])
print('Preflight shape:', preflight_df.shape)
display(preflight_df.head())
display(preflight_df.tail())
print('Selected grid point:')
display(preflight_df[['gfs_grid_lat','gfs_grid_lon','gfs_grid_lat_index','gfs_grid_lon_index']].drop_duplicates())

Preflight shape: (186, 27)


,gfs_init_utc,forecast_hour,valid_time_utc,valid_time_local,gfs_temp_c,gfs_temp_f,gfs_relative_humidity_pct,gfs_dewpoint_c_derived,gfs_dewpoint_f_derived,gfs_u10_mps,...,gfs_precip_rate_kg_m2_s,gfs_precip_rate_mm_hr,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index,data_product_id,data_product_version,snapshot_id,query_id
0,2025-09-16T18:00:00Z,10,2025-09-17T04:00:00Z,2025-09-17T00:00:00-04:00,15.750,60.350,87.0,13.590852,56.463534,1.578125,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
1,2025-09-16T18:00:00Z,11,2025-09-17T05:00:00Z,2025-09-17T01:00:00-04:00,16.250,61.250,84.0,13.542081,56.375747,1.578125,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
2,2025-09-16T18:00:00Z,12,2025-09-17T06:00:00Z,2025-09-17T02:00:00-04:00,16.000,60.800,82.5,13.021173,55.438111,1.453125,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
3,2025-09-16T18:00:00Z,13,2025-09-17T07:00:00Z,2025-09-17T03:00:00-04:00,16.500,61.700,82.5,13.509729,56.317512,2.218750,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
4,2025-09-16T18:00:00Z,14,2025-09-17T08:00:00Z,2025-09-17T04:00:00-04:00,16.625,61.925,82.5,13.631864,56.537356,2.625000,...,0.000017,0.062656,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z


,gfs_init_utc,forecast_hour,valid_time_utc,valid_time_local,gfs_temp_c,gfs_temp_f,gfs_relative_humidity_pct,gfs_dewpoint_c_derived,gfs_dewpoint_f_derived,gfs_u10_mps,...,gfs_precip_rate_kg_m2_s,gfs_precip_rate_mm_hr,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index,data_product_id,data_product_version,snapshot_id,query_id
181,2025-09-16T18:00:00Z,333,2025-09-30T15:00:00Z,2025-09-30T11:00:00-04:00,23.125,73.625,72.5,17.910695,64.239251,-0.625000,...,0.000075,0.271225,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
182,2025-09-16T18:00:00Z,336,2025-09-30T18:00:00Z,2025-09-30T14:00:00-04:00,25.250,77.450,58.0,16.398614,61.517506,0.757812,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
183,2025-09-16T18:00:00Z,339,2025-09-30T21:00:00Z,2025-09-30T17:00:00-04:00,26.375,79.475,54.5,16.468212,61.642782,-0.812500,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
184,2025-09-16T18:00:00Z,342,2025-10-01T00:00:00Z,2025-09-30T20:00:00-04:00,20.125,68.225,94.5,19.213553,66.584395,0.308594,...,0.000168,0.604248,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z
185,2025-09-16T18:00:00Z,345,2025-10-01T03:00:00Z,2025-09-30T23:00:00-04:00,18.875,65.975,92.0,17.546457,63.583623,-2.312500,...,0.000029,0.104284,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,preflight_20250916T18Z


Selected grid point:


,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index
0,36.0,-78.75,216,405


## 6. Cross-check against your saved Herbie checkpoint

In [9]:
def fetch_exact_runs(init_times, prefix='compare'):
    queries=[make_query(x, prefix) for x in init_times]
    response=request_json('POST', FORECAST_ENDPOINT, payload={'queries':queries}, timeout=240)
    return pd.concat([parse_result(r) for r in response['results']], ignore_index=True)

if OLD_HERBIE_CHECKPOINT.exists():
    old=pd.read_csv(OLD_HERBIE_CHECKPOINT)
    print('Found old checkpoint:', OLD_HERBIE_CHECKPOINT)
    print('Rows:', len(old), 'Runs:', old['gfs_init_utc'].nunique())

    required={'gfs_init_utc','forecast_hour','gfs_temp_f'}
    if not required.issubset(old.columns):
        raise RuntimeError(f'Old checkpoint missing expected columns: {sorted(required-set(old.columns))}')

    old['init_norm']=pd.to_datetime(old['gfs_init_utc'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    runs=sorted(old['init_norm'].dropna().unique())

    # You said one checkpoint exists, so this will normally be one request.
    fast_compare=fetch_exact_runs(runs[:10], 'checkpoint_compare')
    fast_compare['init_norm']=pd.to_datetime(fast_compare['gfs_init_utc'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    merged=old.merge(
        fast_compare[['init_norm','forecast_hour','gfs_temp_f','gfs_grid_lat','gfs_grid_lon']],
        on=['init_norm','forecast_hour'], how='inner', suffixes=('_herbie','_fast')
    )
    if len(merged)==0:
        raise RuntimeError('Old checkpoint found, but no overlapping rows could be compared.')

    merged['temp_abs_diff_f']=(merged['gfs_temp_f_herbie']-merged['gfs_temp_f_fast']).abs()
    print('Overlapping rows:', len(merged))
    print('Mean abs difference (°F):', merged['temp_abs_diff_f'].mean())
    print('Max abs difference (°F):', merged['temp_abs_diff_f'].max())
    display(merged.sort_values('temp_abs_diff_f', ascending=False).head(15))

    max_diff=merged['temp_abs_diff_f'].max()
    if max_diff>2.0:
        raise RuntimeError(f'STOP: fast API differs from Herbie by >2°F (max={max_diff:.3f}°F). Send this table back.')
    elif max_diff>0.75:
        print('WARNING: not numerically identical. Differences are under 2°F, but send this table back before modeling.')
    else:
        print('PASS: fast API closely matches the existing Herbie checkpoint.')
else:
    print('Old checkpoint not found at:', OLD_HERBIE_CHECKPOINT)
    print('Because you said one exists, verify this notebook is under notebooks/ and the checkpoint is under data/raw/.')

Found old checkpoint: ../data/raw/_GFS_HISTORICAL_EXPANDED_CHECKPOINT.csv
Rows: 558 Runs: 3
Overlapping rows: 558
Mean abs difference (°F): 0.0628157683341714
Max abs difference (°F): 0.2241101074219216


,gfs_init_utc,forecast_hour,valid_time_utc,valid_time_local,gfs_temp_k,gfs_temp_c,gfs_temp_f_herbie,gfs_dewpoint_k,gfs_dewpoint_c,gfs_dewpoint_f,...,gfs_temp_850_c,gfs_temp_850_f,gfs_grid_lat_herbie,gfs_grid_lon_native,gfs_grid_lon_herbie,init_norm,gfs_temp_f_fast,gfs_grid_lat_fast,gfs_grid_lon_fast,temp_abs_diff_f
165,2021-07-29T18:00:00+00:00,285,2021-08-10T15:00:00+00:00,2021-08-10T11:00:00-04:00,305.274506,32.124506,89.824110,291.200012,18.050012,64.490022,...,20.313135,68.563643,36.0,281.25,-78.75,2021-07-29T18:00:00Z,89.60,36.0,-78.75,0.224110
384,2021-08-12T18:00:00+00:00,22,2021-08-13T16:00:00+00:00,2021-08-13T12:00:00-04:00,307.276276,34.126276,93.427296,293.194061,20.044061,68.079310,...,18.790491,65.822883,36.0,281.25,-78.75,2021-08-12T18:00:00Z,93.65,36.0,-78.75,0.222704
181,2021-07-29T18:00:00+00:00,333,2021-08-12T15:00:00+00:00,2021-08-12T11:00:00-04:00,305.523712,32.373712,90.272682,292.799988,19.649988,67.369978,...,21.346887,70.424397,36.0,281.25,-78.75,2021-07-29T18:00:00Z,90.05,36.0,-78.75,0.222682
273,2021-08-05T18:00:00+00:00,97,2021-08-09T19:00:00+00:00,2021-08-09T15:00:00-04:00,307.268555,34.118555,93.413398,290.700012,17.550012,63.590022,...,18.101709,64.583076,36.0,281.25,-78.75,2021-08-05T18:00:00Z,93.20,36.0,-78.75,0.213398
182,2021-07-29T18:00:00+00:00,336,2021-08-12T18:00:00+00:00,2021-08-12T14:00:00-04:00,310.031677,36.881677,98.387019,291.399994,18.249994,64.849989,...,21.026697,69.848054,36.0,281.25,-78.75,2021-07-29T18:00:00Z,98.60,36.0,-78.75,0.212981
389,2021-08-12T18:00:00+00:00,27,2021-08-13T21:00:00+00:00,2021-08-13T17:00:00-04:00,310.282227,37.132227,98.838008,289.503937,16.353937,61.437086,...,22.381525,72.286744,36.0,281.25,-78.75,2021-08-12T18:00:00Z,99.05,36.0,-78.75,0.211992
294,2021-08-05T18:00:00+00:00,118,2021-08-10T16:00:00+00:00,2021-08-10T12:00:00-04:00,305.515961,32.365961,90.258729,293.000000,19.850000,67.730000,...,18.072534,64.530562,36.0,281.25,-78.75,2021-08-05T18:00:00Z,90.05,36.0,-78.75,0.208729
391,2021-08-12T18:00:00+00:00,29,2021-08-13T23:00:00+00:00,2021-08-13T19:00:00-04:00,306.536774,33.386774,92.096193,292.358490,19.208490,66.575282,...,22.007471,71.613447,36.0,281.25,-78.75,2021-08-12T18:00:00Z,92.30,36.0,-78.75,0.203807
276,2021-08-05T18:00:00+00:00,100,2021-08-09T22:00:00+00:00,2021-08-09T18:00:00-04:00,306.037659,32.887659,91.197786,290.007477,16.857477,62.343458,...,18.912042,66.041676,36.0,281.25,-78.75,2021-08-05T18:00:00Z,91.40,36.0,-78.75,0.202214
498,2021-08-12T18:00:00+00:00,168,2021-08-19T18:00:00+00:00,2021-08-19T14:00:00-04:00,306.260559,33.110559,91.599006,292.402832,19.252832,66.655098,...,16.902704,62.424867,36.0,281.25,-78.75,2021-08-12T18:00:00Z,91.40,36.0,-78.75,0.199006


PASS: fast API closely matches the existing Herbie checkpoint.


## 7. Fast batched downloader with resume support

In [10]:
KEY_COLUMNS=['gfs_init_utc','forecast_hour']


def load_checkpoint(path):
    if not path.exists():
        return pd.DataFrame()
    df=pd.read_csv(path)
    if len(df):
        df['gfs_init_utc']=pd.to_datetime(df['gfs_init_utc'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        df['forecast_hour']=pd.to_numeric(df['forecast_hour'], errors='raise').astype(int)
    return df


def save_checkpoint(df, path):
    df=(df.drop_duplicates(KEY_COLUMNS, keep='last')
          .sort_values(KEY_COLUMNS)
          .reset_index(drop=True))
    df.to_csv(path, index=False)
    return df


def complete_runs(df):
    if len(df)==0:
        return set()
    desired=set(DESIRED_FORECAST_HOURS)
    done=set()
    for init, g in df.groupby('gfs_init_utc'):
        if desired.issubset(set(g['forecast_hour'].astype(int))):
            done.add(init)
    return done


def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

In [11]:
def fetch_batch(init_times, prefix):
    init_times=list(init_times)
    payload={'queries':[make_query(x, prefix) for x in init_times]}
    try:
        response=request_json('POST', FORECAST_ENDPOINT, payload=payload, timeout=240)
    except requests.HTTPError as e:
        status=e.response.status_code if e.response is not None else None
        if status==422 and len(init_times)>1:
            mid=len(init_times)//2
            print(f'Batch of {len(init_times)} received 422; splitting request...')
            a=fetch_batch(init_times[:mid], prefix+'_a')
            b=fetch_batch(init_times[mid:], prefix+'_b')
            return pd.concat([a,b], ignore_index=True)
        raise

    results=response.get('results', [])
    if len(results)!=len(init_times):
        raise RuntimeError(f'Requested {len(init_times)} runs but received {len(results)} results.')
    return pd.concat([parse_result(r) for r in results], ignore_index=True)


def download_collection(init_times, checkpoint_path, prefix, batch_size=BATCH_SIZE):
    checkpoint=load_checkpoint(checkpoint_path)
    done=complete_runs(checkpoint)
    remaining=[pd.Timestamp(x) for x in init_times if iso_z(x) not in done]

    print('Requested runs:', len(init_times))
    print('Already complete:', len(done))
    print('Remaining:', len(remaining))

    batches=list(chunks(remaining, batch_size))
    for i, batch in enumerate(batches, 1):
        print(f'\nBatch {i}/{len(batches)} — {len(batch)} runs')
        for run in batch: print(' ', iso_z(run))
        start=time.time()

        new=fetch_batch(batch, f'{prefix}_b{i}')
        checkpoint=pd.concat([checkpoint,new], ignore_index=True)
        checkpoint=save_checkpoint(checkpoint, checkpoint_path)

        print('Checkpoint saved:', checkpoint_path)
        print('Rows:', f'{len(checkpoint):,}')
        print('Elapsed:', f'{time.time()-start:.1f} sec')

    return checkpoint

## 8. Download all 40 historical runs

In [12]:
historical_fast=download_collection(
    HISTORICAL_RUNS,
    HIST_CHECKPOINT,
    prefix='hist',
    batch_size=BATCH_SIZE,
)

print('\nHistorical checkpoint shape:', historical_fast.shape)
print('Complete runs:', len(complete_runs(historical_fast)))
display(historical_fast.head())
display(historical_fast.tail())

Requested runs: 40
Already complete: 0
Remaining: 40

Batch 1/4 — 10 runs
  2021-07-29T18:00:00Z
  2021-08-05T18:00:00Z
  2021-08-12T18:00:00Z
  2021-08-19T18:00:00Z
  2021-08-26T18:00:00Z
  2021-09-02T18:00:00Z
  2021-09-09T18:00:00Z
  2021-09-16T18:00:00Z
  2022-07-29T18:00:00Z
  2022-08-05T18:00:00Z
Checkpoint saved: ../data/raw/_GFS_FAST_HISTORICAL_CHECKPOINT.csv
Rows: 1,860
Elapsed: 28.1 sec

Batch 2/4 — 10 runs
  2022-08-12T18:00:00Z
  2022-08-19T18:00:00Z
  2022-08-26T18:00:00Z
  2022-09-02T18:00:00Z
  2022-09-09T18:00:00Z
  2022-09-16T18:00:00Z
  2023-07-29T18:00:00Z
  2023-08-05T18:00:00Z
  2023-08-12T18:00:00Z
  2023-08-19T18:00:00Z
Checkpoint saved: ../data/raw/_GFS_FAST_HISTORICAL_CHECKPOINT.csv
Rows: 3,720
Elapsed: 31.6 sec

Batch 3/4 — 10 runs
  2023-08-26T18:00:00Z
  2023-09-02T18:00:00Z
  2023-09-09T18:00:00Z
  2023-09-16T18:00:00Z
  2024-07-29T18:00:00Z
  2024-08-05T18:00:00Z
  2024-08-12T18:00:00Z
  2024-08-19T18:00:00Z
  2024-08-26T18:00:00Z
  2024-09-02T18:00:00Z
Ch

,gfs_init_utc,forecast_hour,valid_time_utc,valid_time_local,gfs_temp_c,gfs_temp_f,gfs_relative_humidity_pct,gfs_dewpoint_c_derived,gfs_dewpoint_f_derived,gfs_u10_mps,...,gfs_precip_rate_kg_m2_s,gfs_precip_rate_mm_hr,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index,data_product_id,data_product_version,snapshot_id,query_id
0,2021-07-29T18:00:00Z,10,2021-07-30T04:00:00Z,2021-07-30T00:00:00-04:00,26.625,79.925,59.5,18.089990,64.561983,2.500000,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b1_20210729T18Z
1,2021-07-29T18:00:00Z,11,2021-07-30T05:00:00Z,2021-07-30T01:00:00-04:00,25.750,78.350,62.5,18.049880,64.489785,2.312500,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b1_20210729T18Z
2,2021-07-29T18:00:00Z,12,2021-07-30T06:00:00Z,2021-07-30T02:00:00-04:00,25.000,77.000,66.5,18.327768,64.989983,2.281250,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b1_20210729T18Z
3,2021-07-29T18:00:00Z,13,2021-07-30T07:00:00Z,2021-07-30T03:00:00-04:00,24.250,75.650,70.5,18.544336,65.379806,1.484375,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b1_20210729T18Z
4,2021-07-29T18:00:00Z,14,2021-07-30T08:00:00Z,2021-07-30T04:00:00-04:00,24.250,75.650,72.5,18.991958,66.185525,2.062500,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b1_20210729T18Z


,gfs_init_utc,forecast_hour,valid_time_utc,valid_time_local,gfs_temp_c,gfs_temp_f,gfs_relative_humidity_pct,gfs_dewpoint_c_derived,gfs_dewpoint_f_derived,gfs_u10_mps,...,gfs_precip_rate_kg_m2_s,gfs_precip_rate_mm_hr,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index,data_product_id,data_product_version,snapshot_id,query_id
7435,2025-09-16T18:00:00Z,333,2025-09-30T15:00:00Z,2025-09-30T11:00:00-04:00,23.125,73.625,72.5,17.910695,64.239251,-0.625000,...,0.000075,0.271225,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b4_20250916T18Z
7436,2025-09-16T18:00:00Z,336,2025-09-30T18:00:00Z,2025-09-30T14:00:00-04:00,25.250,77.450,58.0,16.398614,61.517506,0.757812,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b4_20250916T18Z
7437,2025-09-16T18:00:00Z,339,2025-09-30T21:00:00Z,2025-09-30T17:00:00-04:00,26.375,79.475,54.5,16.468212,61.642782,-0.812500,...,0.000000,0.000000,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b4_20250916T18Z
7438,2025-09-16T18:00:00Z,342,2025-10-01T00:00:00Z,2025-09-30T20:00:00-04:00,20.125,68.225,94.5,19.213553,66.584395,0.308594,...,0.000168,0.604248,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b4_20250916T18Z
7439,2025-09-16T18:00:00Z,345,2025-10-01T03:00:00Z,2025-09-30T23:00:00-04:00,18.875,65.975,92.0,17.546457,63.583623,-2.312500,...,0.000029,0.104284,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,hist_b4_20250916T18Z


## 9. Historical completeness checks and final save

In [13]:
expected={(iso_z(run), int(fxx)) for run in HISTORICAL_RUNS for fxx in DESIRED_FORECAST_HOURS}
actual=set(zip(historical_fast['gfs_init_utc'], historical_fast['forecast_hour'].astype(int)))
missing=sorted(expected-actual)
unexpected=sorted(actual-expected)

print('Expected:', len(expected))
print('Actual:', len(actual))
print('Missing:', len(missing))
print('Unexpected:', len(unexpected))

if missing:
    print('First missing keys:', missing[:20])
    raise RuntimeError('Historical download is incomplete. Rerun the bulk cell; checkpoint resume is automatic.')
if unexpected:
    raise RuntimeError(f'Unexpected keys in checkpoint: {unexpected[:20]}')

core=[
    'gfs_temp_f','gfs_relative_humidity_pct','gfs_u10_mps','gfs_v10_mps',
    'gfs_mslp_hpa','gfs_total_cloud_cover_pct','gfs_precip_rate_mm_hr'
]
print('\nCore missingness (%):')
display((historical_fast[core].isna().mean()*100).round(4).sort_values(ascending=False).to_frame('missing_pct'))

historical_final=(historical_fast.drop_duplicates(KEY_COLUMNS, keep='last')
                  .sort_values(KEY_COLUMNS).reset_index(drop=True))
historical_final.to_csv(HIST_OUTPUT, index=False)
print('\nSaved:', HIST_OUTPUT)
print('Rows:', len(historical_final), 'Runs:', historical_final['gfs_init_utc'].nunique())

Expected: 7440
Actual: 7440
Missing: 0
Unexpected: 0

Core missingness (%):


,missing_pct
gfs_temp_f,0.0
gfs_relative_humidity_pct,0.0
gfs_u10_mps,0.0
gfs_v10_mps,0.0
gfs_mslp_hpa,0.0
gfs_total_cloud_cover_pct,0.0
gfs_precip_rate_mm_hr,0.0



Saved: ../data/raw/NOAA_GFS_FORECAST_HISTORICAL_FAST.csv
Rows: 7440 Runs: 40


## 10. Download the final 2026 run with the same pipeline

In [14]:
future_fast=download_collection(
    [FINAL_GFS_RUN],
    FUTURE_CHECKPOINT,
    prefix='future',
    batch_size=1,
)

expected={(iso_z(FINAL_GFS_RUN), int(fxx)) for fxx in DESIRED_FORECAST_HOURS}
actual=set(zip(future_fast['gfs_init_utc'], future_fast['forecast_hour'].astype(int)))
missing=sorted(expected-actual)

print('Expected future rows:', len(expected))
print('Actual future rows:', len(actual))
print('Missing:', len(missing))

if missing:
    raise RuntimeError(f'Future run incomplete. First missing keys: {missing[:20]}')

future_final=(future_fast.drop_duplicates(KEY_COLUMNS, keep='last')
              .sort_values(KEY_COLUMNS).reset_index(drop=True))
future_final.to_csv(FUTURE_OUTPUT, index=False)

print('Saved:', FUTURE_OUTPUT)
print('Rows:', len(future_final))
display(future_final.head())
display(future_final.tail())

Requested runs: 1
Already complete: 0
Remaining: 1

Batch 1/1 — 1 runs
  2026-09-16T18:00:00Z
Checkpoint saved: ../data/raw/_GFS_FAST_FUTURE_CHECKPOINT.csv
Rows: 186
Elapsed: 2.7 sec
Expected future rows: 186
Actual future rows: 186
Missing: 0
Saved: ../data/raw/NOAA_GFS_FORECAST_FUTURE_FAST.csv
Rows: 186


,gfs_init_utc,forecast_hour,valid_time_utc,valid_time_local,gfs_temp_c,gfs_temp_f,gfs_relative_humidity_pct,gfs_dewpoint_c_derived,gfs_dewpoint_f_derived,gfs_u10_mps,...,gfs_precip_rate_kg_m2_s,gfs_precip_rate_mm_hr,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index,data_product_id,data_product_version,snapshot_id,query_id
0,2026-09-16T18:00:00Z,10,2026-09-17T04:00:00Z,2026-09-17T00:00:00-04:00,20.750,69.350,56.50,11.790468,53.222843,-0.024658,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
1,2026-09-16T18:00:00Z,11,2026-09-17T05:00:00Z,2026-09-17T01:00:00-04:00,20.500,68.900,57.25,11.756859,53.162345,-0.097656,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
2,2026-09-16T18:00:00Z,12,2026-09-17T06:00:00Z,2026-09-17T02:00:00-04:00,19.625,67.325,59.50,11.520687,52.737236,-0.316406,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
3,2026-09-16T18:00:00Z,13,2026-09-17T07:00:00Z,2026-09-17T03:00:00-04:00,19.250,66.650,60.75,11.482491,52.668484,0.617188,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
4,2026-09-16T18:00:00Z,14,2026-09-17T08:00:00Z,2026-09-17T04:00:00-04:00,18.750,65.750,62.50,11.440267,52.592481,0.960938,...,0.0,0.0,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z


,gfs_init_utc,forecast_hour,valid_time_utc,valid_time_local,gfs_temp_c,gfs_temp_f,gfs_relative_humidity_pct,gfs_dewpoint_c_derived,gfs_dewpoint_f_derived,gfs_u10_mps,...,gfs_precip_rate_kg_m2_s,gfs_precip_rate_mm_hr,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index,data_product_id,data_product_version,snapshot_id,query_id
181,2026-09-16T18:00:00Z,333,2026-09-30T15:00:00Z,2026-09-30T11:00:00-04:00,24.750,76.550,80.5,21.167240,70.101031,-2.218750,...,0.000290,1.043701,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
182,2026-09-16T18:00:00Z,336,2026-09-30T18:00:00Z,2026-09-30T14:00:00-04:00,24.000,75.200,85.0,21.321622,70.378920,-0.437500,...,0.000458,1.647949,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
183,2026-09-16T18:00:00Z,339,2026-09-30T21:00:00Z,2026-09-30T17:00:00-04:00,21.625,70.925,94.5,20.703151,69.265672,1.796875,...,0.000273,0.981903,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
184,2026-09-16T18:00:00Z,342,2026-10-01T00:00:00Z,2026-09-30T20:00:00-04:00,20.875,69.575,97.5,20.463973,68.835151,1.390625,...,0.000012,0.041628,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z
185,2026-09-16T18:00:00Z,345,2026-10-01T03:00:00Z,2026-09-30T23:00:00-04:00,19.875,67.775,98.0,19.549392,67.188905,2.093750,...,0.000012,0.041628,36.0,-78.75,216,405,noaa-gfs-forecast,0.2.7,3A7HWV727T9WW44DRYQG,future_b1_20260916T18Z


## 11. Final audit

In [15]:
h=pd.read_csv(HIST_OUTPUT)
f=pd.read_csv(FUTURE_OUTPUT)

summary=pd.DataFrame([
    {
        'dataset': HIST_OUTPUT.name,
        'rows': len(h),
        'runs': h['gfs_init_utc'].nunique(),
        'columns': len(h.columns),
        'min_valid_time': h['valid_time_utc'].min(),
        'max_valid_time': h['valid_time_utc'].max(),
    },
    {
        'dataset': FUTURE_OUTPUT.name,
        'rows': len(f),
        'runs': f['gfs_init_utc'].nunique(),
        'columns': len(f.columns),
        'min_valid_time': f['valid_time_utc'].min(),
        'max_valid_time': f['valid_time_utc'].max(),
    },
])

display(summary)
print('\nSelected GFS grid point(s):')
display(h[['gfs_grid_lat','gfs_grid_lon','gfs_grid_lat_index','gfs_grid_lon_index']].drop_duplicates())
print('\nDone. Keep the old Herbie checkpoint for audit; do not delete it yet.')

,dataset,rows,runs,columns,min_valid_time,max_valid_time
0,NOAA_GFS_FORECAST_HISTORICAL_FAST.csv,7440,40,27,2021-07-30T04:00:00Z,2025-10-01T03:00:00Z
1,NOAA_GFS_FORECAST_FUTURE_FAST.csv,186,1,27,2026-09-17T04:00:00Z,2026-10-01T03:00:00Z



Selected GFS grid point(s):


,gfs_grid_lat,gfs_grid_lon,gfs_grid_lat_index,gfs_grid_lon_index
0,36.0,-78.75,216,405



Done. Keep the old Herbie checkpoint for audit; do not delete it yet.
